In [1]:
import numpy as np
import pandas as pd

from ugdatalab.models.apogee.constants import LABEL_NAMES

In [2]:
# Load all intermediate products
spec_data = np.load("training_spectra.npz", allow_pickle=True)
model_data = np.load("cannon_model.npz", allow_pickle=True)
cv_data = np.load("cv_results.npz", allow_pickle=True)
nn_data = np.load("nn_results.npz", allow_pickle=True)

## Training Set Size and Cuts

Re-derived from `training_spectra.npz`.

In [3]:
N_total = spec_data["flux"].shape[0]
n_pixels = spec_data["wavelength"].shape[0]
n_train = len(model_data["train_idx"])
n_cv = len(model_data["cv_idx"])

training_summary = pd.DataFrame({
    "Quantity": [
        "Total training set (after quality cuts)",
        "Training subset (50%)",
        "Cross-validation subset (50%)",
        "Pixels per spectrum",
        "Wavelength range",
    ],
    "Value": [
        f"{N_total}",
        f"{n_train}",
        f"{n_cv}",
        f"{n_pixels}",
        f"{spec_data['wavelength'][0]:.1f}–{spec_data['wavelength'][-1]:.1f} Å",
    ],
})
training_summary

,Quantity,Value
0,Total training set (after quality cuts),1886
1,Training subset (50%),897
2,Cross-validation subset (50%),897
3,Pixels per spectrum,8575
4,Wavelength range,15100.8–16999.8 Å


## Pixel Bitmask Statistics

Pixels flagged by the APOGEE bitmask have $\sigma = \infty$ after normalization. Re-derived from `training_spectra.npz`.

In [4]:
flux = spec_data["flux"]
error = spec_data["error"]
apogee_ids = spec_data["apogee_ids"]
n_pixels = flux.shape[1]

# Per-star masked pixel counts (inf error or nan flux)
masked_per_star = np.sum(~np.isfinite(flux) | ~np.isfinite(error), axis=1)
frac_per_star = masked_per_star / n_pixels * 100

# Example star
example_id = "2M21235315+1244123"
ex_idx = int(np.where(apogee_ids == example_id)[0][0])

example_bitmask = pd.DataFrame({
    "Quantity": ["Total pixels", "Masked pixels", "Good pixels", "Masked fraction"],
    "Value": [
        f"{n_pixels}",
        f"{masked_per_star[ex_idx]}",
        f"{n_pixels - masked_per_star[ex_idx]}",
        f"{frac_per_star[ex_idx]:.1f}%",
    ],
})
print(f"Example star: {example_id}")
display(example_bitmask)

# Across full training set
dataset_bitmask = pd.DataFrame({
    "Statistic": ["Mean masked pixels", "Median masked pixels", "Min masked pixels", "Max masked pixels", "Mean masked fraction"],
    "Value": [
        f"{np.mean(masked_per_star):.1f}",
        f"{np.median(masked_per_star):.0f}",
        f"{np.min(masked_per_star)}",
        f"{np.max(masked_per_star)}",
        f"{np.mean(frac_per_star):.1f}%",
    ],
})
print(f"\nAcross {len(apogee_ids)} training stars:")
dataset_bitmask

Example star: 2M21235315+1244123


,Quantity,Value
0,Total pixels,8575
1,Masked pixels,1038
2,Good pixels,7537
3,Masked fraction,12.1%



Across 1886 training stars:


,Statistic,Value
0,Mean masked pixels,931.0
1,Median masked pixels,894
2,Min masked pixels,800
3,Max masked pixels,1393
4,Mean masked fraction,10.9%


## Surface Gravity Calculation

Re-derived from fundamental constants.

In [5]:
logg_sun = 4.44  # log g of the Sun in CGS

logg_table = pd.DataFrame({
    "Stage": ["Main sequence", "Pre-He flash (tip RGB)", "Core He burning (red clump)"],
    "R / R_sun": [1, 100, 15],
    "log g": [logg_sun - 2 * np.log10(r) for r in [1, 100, 15]],
})
logg_table

,Stage,R / R_sun,log g
0,Main sequence,1,4.440000
1,Pre-He flash (tip RGB),100,0.440000
2,Core He burning (red clump),15,2.087817


## Cannon Model Parameters

Re-derived from `cannon_model.npz`.

In [6]:
n_terms = model_data["theta"].shape[1]
n_pix = model_data["theta"].shape[0]
n_params_total = (n_terms + 1) * n_pix

cannon_summary = pd.DataFrame({
    "Quantity": [
        "Coefficients per pixel",
        "Number of pixels",
        "Total free parameters",
        "Training chi2_r",
    ],
    "Value": [
        f"{n_terms}",
        f"{n_pix}",
        f"{n_params_total:,}",
        f"{float(model_data['chi2_r']):.3f}",
    ],
})
cannon_summary

,Quantity,Value
0,Coefficients per pixel,21
1,Number of pixels,8575
2,Total free parameters,"188,650"
3,Training chi2_r,1.060


## Cross-Validation Bias and Scatter

Re-derived from `cv_results.npz`.

In [7]:
cannon_fitted = cv_data["fitted_labels"]
true_labels = cv_data["true_labels"]
cannon_resid = cannon_fitted - true_labels

cv_bias_scatter = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Bias (mean offset)": np.mean(cannon_resid, axis=0),
    "Scatter (std)": np.std(cannon_resid, axis=0),
})
cv_bias_scatter

,Label,Bias (mean offset),Scatter (std)
0,TEFF,-0.826737,49.005508
1,LOGG,-0.007898,0.111451
2,FE_H,-0.004868,0.036006
3,MG_FE,0.003713,0.034233
4,SI_FE,0.002083,0.035501


## MCMC Posterior for Mystery Star

Re-derived from the MCMC samples (loaded from the MCMC result, if available, or re-stated from NB 04).

In [8]:
# The MCMC result is produced in NB 04 (terminal analysis).
# To re-derive here, we re-run the MCMC fit.
from pathlib import Path

from ugdatalab.models.apogee import Spectrum
from ugdatalab.methods.cannon import CannonModel
from ugdatalab.methods.cannon_likelihood import CannonLabelLikelihood
from ugdatalab.methods.bayesian.mcmc import nuts_sample

# Reconstruct Cannon model
model = CannonModel(
    theta=model_data["theta"],
    scatter=model_data["scatter"],
    label_names=list(model_data["label_names"]),
    label_means=model_data["label_means"],
    label_stds=model_data["label_stds"],
    wavelength=model_data["wavelength"],
    chi2_r=float(model_data["chi2_r"]),
)

# Read and normalize mystery spectrum
mystery_path = Path("../../course_materials_sp2026/labs/lab_2/mystery_spec_wiped.fits")
continuum_path = Path("continuum_wavelengths.npz")

mystery = Spectrum(mystery_path, continuum_path)
flux_norm = mystery.flux[0]
error_norm = mystery.error[0]

# MCMC fit
lk = CannonLabelLikelihood(x=model.wavelength, y=flux_norm, y_err=error_norm, model=model)
result = nuts_sample(lk, n_steps=2000, n_burn=1000, seed=42)

medians = np.median(result.samples, axis=0)
lo = np.percentile(result.samples, 16, axis=0)
hi = np.percentile(result.samples, 84, axis=0)

mcmc_summary = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Median": medians,
    "16th pct": lo,
    "84th pct": hi,
    "σ (68% CI)": (hi - lo) / 2,
})
mcmc_summary

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [TEFF, LOGG, FE_H, MG_FE, SI_FE]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 5 seconds.


,Label,Median,16th pct,84th pct,σ (68% CI)
0,TEFF,4543.206642,4540.075970,4546.493735,3.208882
1,LOGG,1.954099,1.942944,1.964861,0.010959
2,FE_H,-0.621058,-0.624740,-0.617350,0.003695
3,MG_FE,0.300715,0.297308,0.304026,0.003359
4,SI_FE,0.216299,0.212826,0.219777,0.003475


## Neural Network vs Cannon Comparison

Re-derived from `cv_results.npz` and `nn_results.npz`.

In [9]:
nn_fitted = nn_data["nn_fitted_labels"]
nn_true = nn_data["true_labels"]
nn_resid = nn_fitted - nn_true

comparison = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Cannon bias": np.mean(cannon_resid, axis=0),
    "Cannon scatter": np.std(cannon_resid, axis=0),
    "NN bias": np.mean(nn_resid, axis=0),
    "NN scatter": np.std(nn_resid, axis=0),
})
comparison

,Label,Cannon bias,Cannon scatter,NN bias,NN scatter
0,TEFF,-0.826737,49.005508,47.347534,122.461795
1,LOGG,-0.007898,0.111451,0.098233,0.366343
2,FE_H,-0.004868,0.036006,0.070054,0.157845
3,MG_FE,0.003713,0.034233,-0.016714,0.081786
4,SI_FE,0.002083,0.035501,-0.012036,0.062772
